## Vansh Keserwani
# 230968290

Week-10
Exercise 1: An autonomous delivery robot must navigate a grid-based
warehouse to reach a goal location while avoiding obstacles.
The environment is represented as a grid:
 Each cell = a state
 Actions = {Up, Down, Left, Right}
 Some cells contain obstacles (cannot enter)
 One cell is the goal state
1. Formulate the problem as an RL problem by defining:
 States
 Actions
 Reward function
 Policy
2. Implement Q-Learning:
 Initialize Q-table
 Use ε-greedy exploration
 Update rule:
Q(s,a) = Q(s,a) + α [r + γ max Q(s',a') − Q(s,a)]
3. Train the agent over multiple episodes.

In [9]:
# Exercise 1: Q-Learning for Grid Navigation

import numpy as np
import random

# Grid setup
grid_size = 5
goal = (4, 4)
obstacles = [(1,1), (2,2), (3,1)]

actions = ['up', 'down', 'left', 'right']
alpha = 0.1
gamma = 0.9
epsilon = 0.2
episodes = 200

# Q-table
Q = {}

def get_next_state(state, action):
    x, y = state
    if action == 'up': x -= 1
    elif action == 'down': x += 1
    elif action == 'left': y -= 1
    elif action == 'right': y += 1
    
    if x < 0 or y < 0 or x >= grid_size or y >= grid_size:
        return state
    if (x,y) in obstacles:
        return state
    return (x,y)

def reward(state):
    if state == goal:
        return 10
    return -1

# Training
for _ in range(episodes):
    state = (0,0)
    
    while state != goal:
        if random.random() < epsilon:
            action = random.choice(actions)
        else:
            action = max(actions, key=lambda a: Q.get((state,a),0))
        
        next_state = get_next_state(state, action)
        r = reward(next_state)
        
        old = Q.get((state,action),0)
        future = max([Q.get((next_state,a),0) for a in actions])
        
        Q[(state,action)] = old + alpha * (r + gamma * future - old)
        state = next_state

# Example run
state = (0,0)
path = [state]
while state != goal:
    action = max(actions, key=lambda a: Q.get((state,a),0))
    state = get_next_state(state, action)
    path.append(state)

print("Path to goal:", path)

Path to goal: [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (2, 4), (3, 4), (4, 4)]


Exercise 2: A company wants to choose the best advertisement to show
users. Each ad has an unknown probability of being clicked.
This is modeled as a k-armed bandit problem.
1. Implement:
o ε-greedy strategy
o Upper Confidence Bound (UCB)
2. Simulate:
o 5–10 ads (arms)
o Each with different reward probabilities
3. Track:
o Total reward over time
o Number of times each ad is selected

In [10]:
# Exercise 2: Bandit Problem

import numpy as np

arms = 5
true_probs = np.random.rand(arms)

epsilon = 0.1
steps = 1000

counts = np.zeros(arms)
values = np.zeros(arms)
total_reward = 0

# ε-Greedy
for t in range(steps):
    if np.random.rand() < epsilon:
        action = np.random.randint(arms)
    else:
        action = np.argmax(values)
    
    reward = 1 if np.random.rand() < true_probs[action] else 0
    
    counts[action] += 1
    values[action] += (reward - values[action]) / counts[action]
    total_reward += reward

print("ε-Greedy Total Reward:", total_reward)
print("Arm selection count:", counts)

# UCB
counts = np.zeros(arms)
values = np.zeros(arms)
total_reward = 0

for t in range(1, steps+1):
    ucb = values + np.sqrt(2*np.log(t)/(counts+1e-5))
    action = np.argmax(ucb)
    
    reward = 1 if np.random.rand() < true_probs[action] else 0
    
    counts[action] += 1
    values[action] += (reward - values[action]) / counts[action]
    total_reward += reward

print("UCB Total Reward:", total_reward)
print("Arm selection count:", counts)

ε-Greedy Total Reward: 572
Arm selection count: [914.  23.  28.  13.  22.]
UCB Total Reward: 570
Arm selection count: [692.  38. 193.  25.  52.]


Exercise 3: A system must balance a pole on a moving cart. The goal is
to keep the pole upright as long as possible.Use an environment (e.g., OpenAI Gym or custom simulation)
1. Define:
 State (position, velocity, angle, angular velocity)
 Actions (move left/right)
2.Implement:
 Q-Learning (discretized state) OR
 Deep Q-Network (DQN)
3.Train the agent and plot:
 Episode reward vs episodes

In [11]:
# Exercise 3: CartPole Q-Learning (Fixed using gymnasium)

import gymnasium as gym
import numpy as np

env = gym.make("CartPole-v1")

# Discretization
bins = [10, 10, 10, 10]

def discretize(state):
    upper = env.observation_space.high
    lower = env.observation_space.low
    
    # Fix infinite bounds
    upper[1] = 5
    upper[3] = 5
    lower[1] = -5
    lower[3] = -5
    
    ratios = (state - lower) / (upper - lower)
    ratios = np.clip(ratios, 0, 1)
    
    return tuple((ratios * (np.array(bins) - 1)).astype(int))

# Q-table
Q = {}

alpha = 0.1
gamma = 0.9
epsilon = 0.1
episodes = 200

rewards = []

for ep in range(episodes):
    state, _ = env.reset()
    state = discretize(state)
    
    total_reward = 0
    
    for _ in range(200):
        # ε-greedy
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = max([0,1], key=lambda a: Q.get((state,a),0))
        
        next_state, reward, done, truncated, _ = env.step(action)
        done = done or truncated
        
        next_state = discretize(next_state)
        
        old = Q.get((state, action), 0)
        future = max([Q.get((next_state, a), 0) for a in [0,1]])
        
        Q[(state, action)] = old + alpha * (reward + gamma * future - old)
        
        state = next_state
        total_reward += reward
        
        if done:
            break
    
    rewards.append(total_reward)

# Example output
print("First 10 episode rewards:", rewards[:10])
print("Last 10 episode rewards:", rewards[-10:])

First 10 episode rewards: [10.0, 10.0, 9.0, 11.0, 10.0, 27.0, 9.0, 11.0, 13.0, 27.0]
Last 10 episode rewards: [10.0, 15.0, 9.0, 9.0, 13.0, 10.0, 10.0, 13.0, 12.0, 11.0]


Exercise 4: Design an intelligent traffic signal system that minimizes
congestion at an intersection.
1. Define:
o States: number of vehicles in each lane
o Actions: which signal to turn green
o Reward: negative of total waiting time
2. Implement:
o Q-Learning agent
3. Simulate traffic flow for multiple time steps
4. Evaluate:
o Average waiting time
o Throughput

In [12]:
# Exercise 4: Traffic Signal

import numpy as np
import random

lanes = 2
actions = [0,1]  # which lane gets green

Q = {}
alpha = 0.1
gamma = 0.9
epsilon = 0.2

def get_reward(state):
    return -sum(state)

def next_state(state, action):
    state = list(state)
    
    # cars arrive
    state = [s + np.random.randint(0,3) for s in state]
    
    # green lane clears cars
    state[action] = max(0, state[action] - 2)
    
    return tuple(state)

# Training
for _ in range(200):
    state = (5,5)
    
    for _ in range(50):
        if random.random() < epsilon:
            action = random.choice(actions)
        else:
            action = max(actions, key=lambda a: Q.get((state,a),0))
        
        new_state = next_state(state, action)
        r = get_reward(new_state)
        
        old = Q.get((state,action),0)
        future = max([Q.get((new_state,a),0) for a in actions])
        
        Q[(state,action)] = old + alpha*(r + gamma*future - old)
        state = new_state

# Example run
state = (5,5)
for _ in range(10):
    action = max(actions, key=lambda a: Q.get((state,a),0))
    state = next_state(state, action)
    print("State:", state)

State: (5, 3)
State: (7, 3)
State: (7, 5)
State: (8, 3)
State: (6, 4)
State: (5, 5)
State: (5, 4)
State: (5, 5)
State: (7, 3)
State: (5, 5)


Exercise 5: A medical diagnosis system must determine the probability of
a disease given observed symptoms using probabilistic reasoning.
Given
 Prior probability:
P(Disease)
 Conditional probabilities:
P(Symptom | Disease)
P(Symptom | ¬Disease)
1. Apply Bayes’ Theorem:
P(Disease | Symptom) = [P(Symptom | Disease) * P(Disease)] /
P(Symptom)
2. Extend to multiple symptoms assuming independence:
P(S1, S2 | Disease)
3. Implement a Python program to:
o Take input probabilities
o Compute posterior probability
4. Classify:o
Disease present if probability > threshold

In [14]:
# Exercise 5: Bayesian Diagnosis

def bayes(P_D, P_S_given_D, P_S_given_not_D):
    P_not_D = 1 - P_D
    
    P_S = (P_S_given_D * P_D) + (P_S_given_not_D * P_not_D)
    
    P_D_given_S = (P_S_given_D * P_D) / P_S
    return P_D_given_S

# Example
P_D = 0.01
P_S_given_D = 0.9
P_S_given_not_D = 0.1

result = bayes(P_D, P_S_given_D, P_S_given_not_D)

print("Probability of disease given symptom:", result)

# Classification
threshold = 0.5
if result > threshold:
    print("Disease Present")
else:
    print("Disease Not Present")

Probability of disease given symptom: 0.08333333333333333
Disease Not Present
